# 01 — Provision Infrastructure with Terraform

In this notebook we:
1. Create a Chameleon **lease** reserving 3x m1.large VMs
2. Use **Terraform** to provision those VMs, a private network, a floating IP, and a block storage volume


## Prerequisites

Make sure you have run `00_setup.ipynb` completely before starting this notebook.


In [ ]:
# runs in Chameleon Jupyter environment
# Re-export PATH in each new session
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local
terraform version

## Step 1: Authenticate with Chameleon

Replace `CHI-XXXXXX` with your actual project name, and `YOURNETID` with your net ID.


In [ ]:
# runs in Chameleon Jupyter environment
export OS_AUTH_URL=https://kvm.tacc.chameleoncloud.org:5000/v3
export OS_PROJECT_NAME="CHI-XXXXXX"    export OS_REGION_NAME="KVM@TACC"

## Step 2: Create a reservation lease

We reserve 3 x m1.large VM slots for 12 hours.
Replace `YOURNETID` with your actual net ID.


In [ ]:
# runs in Chameleon Jupyter environment
# replace YOURNETID below
openstack reservation lease create lease_immich_YOURNETID \
  --start-date "$(date -u -d '+10 seconds' '+%Y-%m-%d %H:%M')" \
  --end-date "$(date -u -d '+12 hours' '+%Y-%m-%d %H:%M')" \
  --reservation "resource_type=flavor:instance,flavor_id=$(openstack flavor show m1.large -f value -c id),amount=3"

In [ ]:
# runs in Chameleon Jupyter environment
# replace YOURNETID below — get the reserved flavor UUID
flavor_id=$(openstack reservation lease show lease_immich_YOURNETID -f json -c reservations \
  | jq -r '.reservations[0].flavor_id')
echo $flavor_id

**Copy the UUID printed above — you will paste it into the next cell.**

## Step 3: Set Terraform variables


In [ ]:
# runs in Chameleon Jupyter environment
# Replace each value:
#   YOURNETID       → your net ID (e.g. jc1234)
#   id_rsa_chameleon → name of your SSH key in Chameleon
#   PASTE_UUID_HERE  → the flavor UUID from the cell above

export TF_VAR_suffix=YOURNETID
export TF_VAR_key=id_rsa_chameleon
export TF_VAR_reservation=PASTE_UUID_HERE

echo "suffix:      $TF_VAR_suffix"
echo "key:         $TF_VAR_key"
echo "reservation: $TF_VAR_reservation"

## Step 4: Initialize and validate Terraform

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH

# Unset OS_ vars — Terraform reads credentials from clouds.yaml instead
unset $(set | grep -o "^OS_[A-Za-z0-9_]*")

cd /work/immich-iac/tf/kvm
terraform init

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
cd /work/immich-iac/tf/kvm
terraform validate

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
cd /work/immich-iac/tf/kvm
terraform plan

## Step 5: Apply — create the VMs

This takes about 3–5 minutes.  
When it finishes, **write down the `node1_floating_ip`** from the output — you need it in every notebook after this.


In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
cd /work/immich-iac/tf/kvm
terraform apply -auto-approve

**Note the floating IP printed in the output above.**

From the KVM@TACC Horizon GUI, check the list of compute instances and confirm your 3 nodes appear.

---
**Continue to `02_install_k8s.ipynb`**
